# Bronze layer
## Loading data from volumes to our database

Defining ingestion configuration

In [ ]:
%run ../utils/logger

In [ ]:
%run ../utils/dataquality

In [0]:
INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/raw_sources/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/raw_sources/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]

Iterating through tables and writing them to delta tables

In [ ]:
for item in INGESTION_CONFIG:
    table_name = f"bronze.{item['table']}"
    log_event("bronze", table_name, "START", f"Ingesting from {item['source']}")

    corrected_path = item['path'].replace('/raw_sources/', '/source_systems/')
    df = spark.read.csv(corrected_path, header=True, inferSchema=True)

    row_count = check_not_empty(df, table_name)
    df.write.mode("overwrite").format("delta").saveAsTable(table_name)

    log_event("bronze", table_name, "SUCCESS", "Ingestion complete", rows=row_count)